# Gift Concierge — Person 2: AI Personalization Engine

## 1. Install Dependencies

In [ ]:
%pip -q install -U groq pydantic

## 2. Mount Drive and create shared project folders

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import datetime
import re
from typing import List, Dict, Any

PROJECT_DIR = Path("/content/drive/MyDrive/Sensory_Gift_Concierge/Person_2_AI")
INPUT_DIR = PROJECT_DIR / "inputs"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TEST_DIR = PROJECT_DIR / "test_runs"
REVISED_TEST_DIR = TEST_DIR / "revised_schema"

for folder in [INPUT_DIR, OUTPUT_DIR, TEST_DIR, REVISED_TEST_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Inputs:", INPUT_DIR)
print("Outputs:", OUTPUT_DIR)
print("Test runs:", TEST_DIR)

## 3. Load Groq API key securely

In [ ]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY was not found. Add it in Colab Secrets and enable notebook access."
    )

client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "openai/gpt-oss-20b"

print("Groq client initialized.")
print("Model configured:", MODEL_NAME)

# 4. Person 1 -> Person 2 input contract

In [ ]:
{
  "relationship": "best friend",
  "personality": ["creative", "emotional"],
  "memories": "We travelled together and took photographs.",
  "interests": ["photography", "travel"],
  "sensory_preferences": ["sight", "touch"],
  "desired_emotion": "nostalgia",
  "occasion": "birthday",
  "budget_inr": 1500
}

## 4. Sample input contract from Person 1

In [ ]:
sample_questionnaire = {
    "relationship": "best friend",
    "personality": ["creative", "emotional"],
    "memories": "We travelled together and took photographs.",
    "interests": ["photography", "travel"],
    "sensory_preferences": ["sight", "touch"],
    "desired_emotion": "nostalgia",
    "occasion": "birthday",
    "budget_inr": 1500
}

sample_input_path = INPUT_DIR / "sample_questionnaire.json"
with open(sample_input_path, "w", encoding="utf-8") as f:
    json.dump(sample_questionnaire, f, ensure_ascii=False, indent=2)

print("Sample questionnaire saved:", sample_input_path)
print(json.dumps(sample_questionnaire, ensure_ascii=False, indent=2))

# 5. Validate questionnaire input

In [ ]:
REQUIRED_INPUT_FIELDS = [
    "relationship",
    "personality",
    "memories",
    "interests",
    "sensory_preferences",
    "desired_emotion",
    "occasion",
    "budget_inr"
]

LIST_INPUT_FIELDS = ["personality", "interests", "sensory_preferences"]

def validate_questionnaire(questionnaire: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(questionnaire, dict):
        raise TypeError("Questionnaire must be a Python dictionary.")

    missing = [key for key in REQUIRED_INPUT_FIELDS if key not in questionnaire]
    if missing:
        raise ValueError(f"Missing required questionnaire fields: {missing}")

    for field in ["relationship", "memories", "desired_emotion", "occasion"]:
        value = questionnaire[field]
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"'{field}' must be a non-empty string.")

    for field in LIST_INPUT_FIELDS:
        value = questionnaire[field]
        if not isinstance(value, list):
            raise TypeError(f"'{field}' must be a list.")
        if any(not isinstance(item, str) or not item.strip() for item in value):
            raise ValueError(f"Every item in '{field}' must be a non-empty string.")

    budget = questionnaire["budget_inr"]
    if isinstance(budget, bool) or not isinstance(budget, int) or budget <= 0:
        raise ValueError("'budget_inr' must be a positive integer.")

    return questionnaire

validate_questionnaire(sample_questionnaire)
print("Sample input passed local validation.")

## 6. Define the structured output schema

In [ ]:

from pydantic import BaseModel, Field, ConfigDict
from typing import List


# 1. Categorized matching tags
class MatchingTags(BaseModel):
    model_config = ConfigDict(extra="forbid")

    interest_tags: List[str] = Field(default_factory=list)
    emotion_tags: List[str] = Field(default_factory=list)
    sensory_tags: List[str] = Field(default_factory=list)
    personality_tags: List[str] = Field(default_factory=list)
    style_tags: List[str] = Field(default_factory=list)
    occasion_tags: List[str] = Field(default_factory=list)


# 2. Recipient profile
class RecipientProfile(BaseModel):
    model_config = ConfigDict(extra="forbid")

    relationship: str
    personality_descriptors: List[str]
    interests: List[str]
    emotional_theme: str
    sensory_preferences: List[str]
    gift_style: str
    matching_tags: MatchingTags
    budget_inr: int
    profile_summary: str


# 3. Complete personalization result
class PersonalizationResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    recipient_profile: RecipientProfile
    recommendation_directions: List[str]
    match_rationales: List[str]
    missing_information: List[str]
    safety_note: str


# 4. Verify model definitions
print("SUCCESS: All Pydantic models defined.")

print("\nRecipient profile fields:")
print(list(RecipientProfile.model_fields.keys()))

print("\nMatching tag fields:")
print(list(MatchingTags.model_fields.keys()))

print("\nPersonalization result fields:")
print(list(PersonalizationResult.model_fields.keys()))


# 7. Define Groq strict JSON Schema

In [ ]:
GIFT_SCHEMA = {
    "type": "object",
    "properties": {
        "recipient_profile": {
            "type": "object",
            "properties": {
                "relationship": {"type": "string"},
                "personality_descriptors": {
                    "type": "array", "items": {"type": "string"}
                },
                "interests": {
                    "type": "array", "items": {"type": "string"}
                },
                "emotional_theme": {"type": "string"},
                "sensory_preferences": {
                    "type": "array", "items": {"type": "string"}
                },
                "gift_style": {"type": "string"},
                "matching_tags": {
                    "type": "object",
                    "properties": {
                        "interest_tags": {"type": "array", "items": {"type": "string"}},
                        "emotion_tags": {"type": "array", "items": {"type": "string"}},
                        "sensory_tags": {"type": "array", "items": {"type": "string"}},
                        "personality_tags": {"type": "array", "items": {"type": "string"}},
                        "style_tags": {"type": "array", "items": {"type": "string"}},
                        "occasion_tags": {"type": "array", "items": {"type": "string"}}
                    },
                    "required": [
                        "interest_tags", "emotion_tags", "sensory_tags",
                        "personality_tags", "style_tags", "occasion_tags"
                    ],
                    "additionalProperties": False
                },
                "budget_inr": {"type": "integer"},
                "profile_summary": {"type": "string"}
            },
            "required": [
                "relationship", "personality_descriptors", "interests",
                "emotional_theme", "sensory_preferences", "gift_style",
                "matching_tags", "budget_inr", "profile_summary"
            ],
            "additionalProperties": False
        },
        "recommendation_directions": {
            "type": "array", "items": {"type": "string"}
        },
        "match_rationales": {
            "type": "array", "items": {"type": "string"}
        },
        "missing_information": {
            "type": "array", "items": {"type": "string"}
        },
        "safety_note": {"type": "string"}
    },
    "required": [
        "recipient_profile", "recommendation_directions",
        "match_rationales", "missing_information", "safety_note"
    ],
    "additionalProperties": False
}

# Basic structural consistency checks
assert set(MatchingTags.model_fields.keys()) == set(
    GIFT_SCHEMA["properties"]["recipient_profile"]["properties"]["matching_tags"]["properties"].keys()
)
assert "matching_tags" in RecipientProfile.model_fields
print("Groq JSON schema defined and tag fields aligned.")

# 8. System instructions for the personalization engine

In [ ]:

# ============================================================
# CELL 8: REVISED SYSTEM INSTRUCTIONS, GROQ SCHEMA AND FUNCTION
# ============================================================

import json
from pydantic import ValidationError

# ------------------------------------------------------------
# 1. SYSTEM INSTRUCTIONS
# ------------------------------------------------------------

SYSTEM_INSTRUCTIONS = """
You are the AI Personalization Engine for an AI-Guided Sensory Gift Concierge.

Your responsibility is to interpret the recipient questionnaire and create
a structured recipient profile for downstream product matching.

IMPORTANT ROLE BOUNDARY:
- You interpret the recipient's personality, interests and emotional needs.
- You identify sensory preferences and recommendation directions.
- You do NOT select actual products, brands, merchants or prices.
- Actual product selection is handled by Person 3.

OUTPUT RULES:

1. Preserve the recipient's relationship, personality, interests,
   emotional theme, sensory preferences and budget.

2. matching_tags must contain exactly these six categories:
   - interest_tags
   - emotion_tags
   - sensory_tags
   - personality_tags
   - style_tags
   - occasion_tags

3. INTEREST TAGS:
   Include every interest explicitly supplied in the questionnaire.
   You may add closely related tags, but do not omit the original interests.

4. EMOTION TAGS:
   Include the exact desired emotion supplied by the user.
   You may add relevant emotional variations.

5. SENSORY TAGS:
   Include every sensory preference explicitly supplied by the user.
   Use the original terms: sight, sound, smell and touch.
   Do not replace the original terms with synonyms.

6. PERSONALITY TAGS:
   Include relevant personality descriptors from the questionnaire.

7. STYLE TAGS:
   Describe the broad gift style supported by the available information.
   Do not assume handmade, luxury, minimalist or other styles without evidence.
   gift_style must never be empty.
   If evidence is insufficient, use "versatile".

8. OCCASION TAGS:
   Include the occasion supplied in the questionnaire.

9. BUDGET:
   Preserve budget_inr exactly as supplied.
   Do not increase, decrease or reinterpret it.

10. RECOMMENDATION DIRECTIONS:
    Provide broad recommendation categories and directions.
    Do not mention specific products, brands, merchants or prices.

11. MATCH RATIONALES:
    Explain why the recommendation directions match the recipient profile.

12. MISSING INFORMATION:
    Mention only information that is genuinely important but unavailable.

13. Do not invent memories, interests, preferences or personality traits.

14. Return valid JSON matching the supplied JSON schema.
    Do not include Markdown, explanations or text outside the JSON.
"""


# ------------------------------------------------------------
# 2. REVISED GROQ JSON SCHEMA
# ------------------------------------------------------------

GIFT_SCHEMA = {
    "type": "object",
    "properties": {
        "recipient_profile": {
            "type": "object",
            "properties": {
                "relationship": {"type": "string"},
                "personality_descriptors": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "interests": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "emotional_theme": {"type": "string"},
                "sensory_preferences": {
                    "type": "array",
                    "items": {"type": "string"}
                },
                "gift_style": {"type": "string"},

                # NEW STRUCTURED MATCHING TAGS
                "matching_tags": {
                    "type": "object",
                    "properties": {
                        "interest_tags": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "emotion_tags": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "sensory_tags": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "personality_tags": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "style_tags": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "occasion_tags": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "interest_tags",
                        "emotion_tags",
                        "sensory_tags",
                        "personality_tags",
                        "style_tags",
                        "occasion_tags"
                    ],
                    "additionalProperties": False
                },

                "budget_inr": {"type": "integer"},
                "profile_summary": {"type": "string"}
            },
            "required": [
                "relationship",
                "personality_descriptors",
                "interests",
                "emotional_theme",
                "sensory_preferences",
                "gift_style",
                "matching_tags",
                "budget_inr",
                "profile_summary"
            ],
            "additionalProperties": False
        },

        "recommendation_directions": {
            "type": "array",
            "items": {"type": "string"}
        },
        "match_rationales": {
            "type": "array",
            "items": {"type": "string"}
        },
        "missing_information": {
            "type": "array",
            "items": {"type": "string"}
        },
        "safety_note": {"type": "string"}
    },
    "required": [
        "recipient_profile",
        "recommendation_directions",
        "match_rationales",
        "missing_information",
        "safety_note"
    ],
    "additionalProperties": False
}


# ------------------------------------------------------------
# 3. VERIFY SCHEMA BEFORE MAKING AN API REQUEST
# ------------------------------------------------------------

assert "matching_tags" in GIFT_SCHEMA["properties"]["recipient_profile"]["properties"]

assert (
    "product_matching_tags"
    not in GIFT_SCHEMA["properties"]["recipient_profile"]["properties"]
)

assert set(
    GIFT_SCHEMA["properties"]["recipient_profile"]["properties"]["matching_tags"]["properties"]
) == {
    "interest_tags",
    "emotion_tags",
    "sensory_tags",
    "personality_tags",
    "style_tags",
    "occasion_tags"
}

print("Groq schema verification successful.")
print("New matching_tags structure detected.")
print("Old product_matching_tags field is absent.")


# ------------------------------------------------------------
# 4. QUESTIONNAIRE VALIDATION
# ------------------------------------------------------------

def validate_questionnaire(questionnaire):

    required_fields = [
        "relationship",
        "personality",
        "memories",
        "interests",
        "sensory_preferences",
        "desired_emotion",
        "occasion",
        "budget_inr"
    ]

    missing = [
        field for field in required_fields
        if field not in questionnaire
    ]

    if missing:
        raise ValueError(f"Missing questionnaire fields: {missing}")

    if not isinstance(questionnaire["budget_inr"], int):
        raise ValueError("budget_inr must be an integer.")

    if questionnaire["budget_inr"] <= 0:
        raise ValueError("budget_inr must be greater than zero.")

    return True



# ============================================================
# PERSONALIZATION FUNCTION - TOKEN LIMIT CORRECTION
# ============================================================

import json

def personalize_recipient(questionnaire):

    # 1. Validate input
    validate_questionnaire(questionnaire)

    # 2. Send request to Groq
    response = client.chat.completions.create(
        model=MODEL_NAME,

        messages=[
            {
                "role": "system",
                "content": SYSTEM_INSTRUCTIONS
            },
            {
                "role": "user",
                "content": (
                    "Generate the recipient personalization result "
                    "using the required JSON schema.\n\n"
                    "Keep descriptions concise. Avoid unnecessary "
                    "explanations and repeated information.\n\n"
                    + json.dumps(questionnaire, ensure_ascii=False)
                )
            }
        ],

        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "gift_personalization_result",
                "strict": True,
                "schema": GIFT_SCHEMA
            }
        },

        # Increased limit for reasoning + structured JSON output
        max_completion_tokens=8192,

        temperature=0.2
    )

    # 3. Read response
    response_text = response.choices[0].message.content

    if not response_text:
        raise RuntimeError("Groq returned an empty response.")

    # 4. Validate against Pydantic schema
    result = PersonalizationResult.model_validate_json(response_text)

    # 5. Preserve original questionnaire information
    profile = result.recipient_profile

    profile.relationship = questionnaire["relationship"]
    profile.personality_descriptors = list(questionnaire["personality"])
    profile.interests = list(questionnaire["interests"])
    profile.emotional_theme = questionnaire["desired_emotion"]
    profile.sensory_preferences = list(questionnaire["sensory_preferences"])
    profile.budget_inr = questionnaire["budget_inr"]

    tags = profile.matching_tags

    # 6. Ensure all supplied interests are included
    for item in questionnaire["interests"]:
        if item not in tags.interest_tags:
            tags.interest_tags.append(item)

    # 7. Ensure desired emotion is included
    emotion = questionnaire["desired_emotion"]

    if emotion not in tags.emotion_tags:
        tags.emotion_tags.append(emotion)

    # 8. Ensure sensory preferences are included
    for item in questionnaire["sensory_preferences"]:
        if item not in tags.sensory_tags:
            tags.sensory_tags.append(item)

    # 9. Ensure personality descriptors are included
    for item in questionnaire["personality"]:
        if item not in tags.personality_tags:
            tags.personality_tags.append(item)

    # 10. Ensure occasion is included
    occasion = questionnaire["occasion"]

    if occasion not in tags.occasion_tags:
        tags.occasion_tags.append(occasion)

    # 11. Ensure gift style is not empty
    if not profile.gift_style or not profile.gift_style.strip():
        profile.gift_style = "versatile"

    if profile.gift_style not in tags.style_tags:
        tags.style_tags.append(profile.gift_style)

    return result


print("Personalization function updated.")
print("Completion token limit: 8192")
print("Structured matching tags: enabled.")


# ------------------------------------------------------------
# 6. SAMPLE QUESTIONNAIRE
# ------------------------------------------------------------

sample_questionnaire = {
    "relationship": "best friend",
    "personality": ["creative", "emotional"],
    "memories": "We travelled together and took photographs.",
    "interests": ["photography", "travel"],
    "sensory_preferences": ["sight", "touch"],
    "desired_emotion": "nostalgia",
    "occasion": "birthday",
    "budget_inr": 1500
}


# ------------------------------------------------------------
# 7. RUN ONE TEST
# ------------------------------------------------------------

print("\nValidating sample questionnaire...")
print("Questionnaire valid:", validate_questionnaire(sample_questionnaire))

print("\nCalling Groq personalization engine...")

result = personalize_recipient(sample_questionnaire)

print("\nPERSONALIZATION RESULT")
print("=" * 60)
print(result.model_dump_json(indent=2))

print("\nCell 8 executed successfully.")


## 9. Call Groq and validate the structured response

In [ ]:
def personalize_recipient(questionnaire: Dict[str, Any]) -> PersonalizationResult:
    validate_questionnaire(questionnaire)

    user_prompt = (
        "Questionnaire JSON:\\n"
        + json.dumps(questionnaire, ensure_ascii=False)
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_INSTRUCTIONS},
                {"role": "user", "content": user_prompt}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "gift_personalization",
                    "strict": True,
                    "schema": GIFT_SCHEMA
                }
            },
            temperature=0.2,
            max_tokens=1500
        )

        response_text = response.choices[0].message.content
        if not response_text:
            raise RuntimeError("Groq returned an empty response.")

        result = PersonalizationResult.model_validate_json(response_text)

        # Preserve authoritative questionnaire values.
        result.recipient_profile.budget_inr = questionnaire["budget_inr"]
        result.recipient_profile.relationship = questionnaire["relationship"]
        result.recipient_profile.personality_descriptors = list(questionnaire["personality"])
        result.recipient_profile.interests = list(questionnaire["interests"])
        result.recipient_profile.emotional_theme = questionnaire["desired_emotion"]
        result.recipient_profile.sensory_preferences = list(questionnaire["sensory_preferences"])

        # Enforce explicit coverage of the original values in tag groups.
        tags = result.recipient_profile.matching_tags
        for value in questionnaire["interests"]:
            if value.lower() not in [x.lower() for x in tags.interest_tags]:
                tags.interest_tags.append(value)

        emotion = questionnaire["desired_emotion"]
        if emotion.lower() not in [x.lower() for x in tags.emotion_tags]:
            tags.emotion_tags.append(emotion)

        for value in questionnaire["sensory_preferences"]:
            if value.lower() not in [x.lower() for x in tags.sensory_tags]:
                tags.sensory_tags.append(value)

        for value in questionnaire["personality"]:
            if value.lower() not in [x.lower() for x in tags.personality_tags]:
                tags.personality_tags.append(value)

        occasion = questionnaire["occasion"]
        if occasion.lower() not in [x.lower() for x in tags.occasion_tags]:
            tags.occasion_tags.append(occasion)

        # Non-empty fallback for gift style.
        if not result.recipient_profile.gift_style.strip():
            result.recipient_profile.gift_style = "versatile"
        if result.recipient_profile.gift_style.lower() not in [
            x.lower() for x in tags.style_tags
        ]:
            tags.style_tags.append(result.recipient_profile.gift_style.lower())

        return result

    except Exception as exc:
        # Do not print credentials or request headers.
        error_type = type(exc).__name__
        raise RuntimeError(
            f"Personalization request failed ({error_type}). "
            "Check model availability, Groq limits and the API response. "
            "Do not share your API key."
        ) from exc

print("Personalization function defined. No API call has been made yet.")

## 10. Run one controlled smoke test

In [ ]:
# One API call:
smoke_test_result = personalize_recipient(sample_questionnaire)

print(smoke_test_result.model_dump_json(indent=2))

# 11. Save the latest result and a versioned handoff file

In [ ]:
def save_personalization_result(result: PersonalizationResult, prefix: str = "personalization"):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    data = result.model_dump()

    versioned_path = OUTPUT_DIR / f"{prefix}_{timestamp}.json"
    latest_path = OUTPUT_DIR / "latest_personalization_result.json"

    for path in [versioned_path, latest_path]:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    print("Versioned output:", versioned_path)
    print("Latest handoff:", latest_path)
    return versioned_path, latest_path

smoke_test_paths = save_personalization_result(smoke_test_result, "smoke_test")

# 12. Load a questionnaire JSON from Person 1

In [ ]:
PERSON_1_INPUT_PATH = (
    Path("/content/drive/MyDrive/Sensory_Gift_Concierge")
    / "Person_1_Frontend" / "outputs" / "questionnaire_response.json"
)

def load_questionnaire_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Questionnaire file not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    validate_questionnaire(data)
    return data

if PERSON_1_INPUT_PATH.exists():
    person1_questionnaire = load_questionnaire_json(PERSON_1_INPUT_PATH)
    print("Loaded Person 1 questionnaire:", PERSON_1_INPUT_PATH)
    print(json.dumps(person1_questionnaire, ensure_ascii=False, indent=2))
else:
    person1_questionnaire = None
    print("Person 1 file not found yet. This is expected until Person 1 shares the file.")

# 13. Process a real Person 1 questionnaire (run only when the file exists)

In [ ]:
# Uncomment and run when Person 1's file is ready.
#
# person1_result = personalize_recipient(person1_questionnaire)
# save_personalization_result(person1_result, "person1_personalization")
# print(person1_result.model_dump_json(indent=2))

# 14. Synthetic evaluation test cases

In [ ]:
test_cases = [
    {
        "test_id": "TC01",
        "relationship": "best friend",
        "personality": ["creative", "emotional"],
        "memories": "We travelled together and took many photographs.",
        "interests": ["photography", "travel"],
        "sensory_preferences": ["sight", "touch"],
        "desired_emotion": "nostalgia",
        "occasion": "birthday",
        "budget_inr": 1500
    },
    {
        "test_id": "TC02",
        "relationship": "mother",
        "personality": ["caring", "calm"],
        "memories": "We used to spend time gardening together.",
        "interests": ["gardening", "reading"],
        "sensory_preferences": ["smell", "touch"],
        "desired_emotion": "relaxation",
        "occasion": "birthday",
        "budget_inr": 2000
    },
    {
        "test_id": "TC03",
        "relationship": "brother",
        "personality": ["adventurous", "curious"],
        "memories": "We enjoy exploring new places together.",
        "interests": ["gaming", "technology", "travel"],
        "sensory_preferences": ["sight", "sound"],
        "desired_emotion": "adventure",
        "occasion": "birthday",
        "budget_inr": 2500
    },
    {
        "test_id": "TC04",
        "relationship": "sister",
        "personality": ["creative", "determined"],
        "memories": "She is preparing for an important academic milestone.",
        "interests": ["art", "journaling"],
        "sensory_preferences": ["sight", "touch"],
        "desired_emotion": "encouragement",
        "occasion": "graduation",
        "budget_inr": 1000
    },
    {
        "test_id": "TC05",
        "relationship": "father",
        "personality": ["practical", "thoughtful"],
        "memories": "He has always supported our family.",
        "interests": ["reading", "music"],
        "sensory_preferences": ["sound", "touch"],
        "desired_emotion": "appreciation",
        "occasion": "father's day",
        "budget_inr": 3000
    }
]

print("Defined", len(test_cases), "synthetic test cases.")
print("No API calls have been made in this cell.")

# 15. Enhanced evaluation for the revised schema

In [ ]:
def _normalized_set(values):
    return {str(value).strip().lower() for value in values if str(value).strip()}

def evaluate_revised_output(questionnaire: Dict[str, Any], result: PersonalizationResult):
    profile = result.recipient_profile
    tags = profile.matching_tags

    checks = {
        "Required profile fields are non-empty": all([
            bool(profile.relationship.strip()),
            bool(profile.emotional_theme.strip()),
            bool(profile.gift_style.strip()),
            bool(profile.profile_summary.strip())
        ]),
        "All interests explicitly represented": _normalized_set(questionnaire["interests"]).issubset(
            _normalized_set(tags.interest_tags)
        ),
        "Desired emotion explicitly represented": questionnaire["desired_emotion"].strip().lower() in
            _normalized_set(tags.emotion_tags),
        "All sensory preferences explicitly represented": _normalized_set(questionnaire["sensory_preferences"]).issubset(
            _normalized_set(tags.sensory_tags)
        ),
        "All personality descriptors represented": _normalized_set(questionnaire["personality"]).issubset(
            _normalized_set(tags.personality_tags)
        ),
        "Occasion represented": questionnaire["occasion"].strip().lower() in
            _normalized_set(tags.occasion_tags),
        "Budget preserved": profile.budget_inr == questionnaire["budget_inr"],
        "Recommendation directions generated": len(result.recommendation_directions) > 0,
        "Missing information is a list": isinstance(result.missing_information, list)
    }

    return checks

print("Revised evaluation function defined.")

# 16. Run one selected test case

In [ ]:
# Change this index to select a different test case.
TEST_CASE_INDEX = 3  # Default: TC04

selected_test = test_cases[TEST_CASE_INDEX]
selected_test_id = selected_test["test_id"]
selected_questionnaire = {
    key: value for key, value in selected_test.items() if key != "test_id"
}

print("Selected:", selected_test_id)
print(json.dumps(selected_questionnaire, ensure_ascii=False, indent=2))

# One API call when this cell is run:
selected_result = personalize_recipient(selected_questionnaire)

print("\\nGENERATED OUTPUT:")
print(selected_result.model_dump_json(indent=2))

selected_evaluation = evaluate_revised_output(
    selected_questionnaire,
    selected_result
)

print("\\nEVALUATION:")
for name, passed in selected_evaluation.items():
    print(("PASS" if passed else "FAIL") + " - " + name)

print(
    "Score:",
    sum(selected_evaluation.values()),
    "/",
    len(selected_evaluation)
)

# 17. Save a revised-schema test record

In [ ]:
def save_revised_test_record(
    test_id: str,
    questionnaire: Dict[str, Any],
    result: PersonalizationResult,
    evaluation: Dict[str, bool]
):
    record = {
        "test_id": test_id,
        "schema_version": "2.0",
        "input": questionnaire,
        "generated_output": result.model_dump(),
        "automatic_evaluation": evaluation,
        "passed_checks": sum(evaluation.values()),
        "total_checks": len(evaluation),
        "evaluation_note": (
            "Rule-based structural and tag coverage checks. "
            "Not a measure of real-world recommendation accuracy."
        ),
        "created_at": datetime.datetime.now().isoformat()
    }

    path = REVISED_TEST_DIR / f"{test_id}_evaluation.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)

    print("Revised test record saved:", path)
    return path

# Run after Section 16:
# revised_record_path = save_revised_test_record(
#     selected_test_id,
#     selected_questionnaire,
#     selected_result,
#     selected_evaluation
# )

# 18. Export a Person 3 handoff summary

In [ ]:
def create_person3_handoff(result: PersonalizationResult):
    profile = result.recipient_profile

    handoff = {
        "schema_version": "2.0",
        "recipient_profile": profile.model_dump(),
        "recommendation_directions": result.recommendation_directions,
        "match_rationales": result.match_rationales,
        "budget_inr": profile.budget_inr,
        "ownership_note": (
            "Person 2 supplies recipient interpretation and matching tags. "
            "Person 3 selects and verifies actual catalogue products and prices."
        )
    }

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = OUTPUT_DIR / f"person3_handoff_{timestamp}.json"
    latest_path = OUTPUT_DIR / "latest_person3_handoff.json"

    for target in [path, latest_path]:
        with open(target, "w", encoding="utf-8") as f:
            json.dump(handoff, f, ensure_ascii=False, indent=2)

    print("Handoff saved:", path)
    print("Latest handoff:", latest_path)
    return path, latest_path

# Run after you have a successful result:
# handoff_paths = create_person3_handoff(selected_result)

# 19. Local validation tests (no API calls)

In [ ]:
def run_local_input_validation_tests():
    tests = []

    invalid_missing = dict(sample_questionnaire)
    invalid_missing.pop("relationship")
    tests.append(("Missing field rejected", invalid_missing))

    invalid_budget = dict(sample_questionnaire)
    invalid_budget["budget_inr"] = -100
    tests.append(("Negative budget rejected", invalid_budget))

    invalid_list = dict(sample_questionnaire)
    invalid_list["interests"] = "photography"
    tests.append(("Wrong list type rejected", invalid_list))

    for label, payload in tests:
        try:
            validate_questionnaire(payload)
            print("FAIL -", label)
        except (ValueError, TypeError):
            print("PASS -", label)

run_local_input_validation_tests()

In [ ]:
print("ACTIVE MODEL:", MODEL_NAME)

profile_schema = GIFT_SCHEMA["properties"]["recipient_profile"]["properties"]

print("\nPydantic fields:")
print(list(RecipientProfile.model_fields.keys()))

print("\nActive Groq schema fields:")
print(list(profile_schema.keys()))

print("\nTag field present:", "matching_tags" in profile_schema)
print("Old field present:", "product_matching_tags" in profile_schema)

assert "matching_tags" in profile_schema, "Old or incorrect schema is active."
assert "product_matching_tags" not in profile_schema, "Old tag field still exists."

print("\nPASS: Active schema is using the revised structure.")

In [ ]:

# ============================================================
# UNBOXJOY - GITHUB CONNECTION AND REPOSITORY SETUP
# Safe to run again: reuses an existing clone.
# ============================================================

import os
import stat
import subprocess
import requests
from pathlib import Path
from google.colab import userdata

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

REPO_URL = "https://github.com/YashikaGupta09/UnboxJoy.git"
REPO_API = "https://api.github.com/repos/YashikaGupta09/UnboxJoy"

BRANCH_NAME = "SakshiJha-AI"
CLONE_PATH = Path("/content/UnboxJoy")
ASKPASS_PATH = Path("/content/github_askpass.sh")

# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def run_git(command, env=None, check=True):
    result = subprocess.run(
        ["git"] + command,
        cwd=CLONE_PATH,
        env=env,
        capture_output=True,
        text=True
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Git command failed: {' '.join(command)}\n"
            f"{result.stderr}"
        )

    return result

# ------------------------------------------------------------
# STEP 1: READ GITHUB TOKEN FROM COLAB SECRETS
# ------------------------------------------------------------

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

    if not GITHUB_TOKEN:
        raise ValueError("GITHUB_TOKEN is empty.")

except Exception as e:
    raise RuntimeError(
        "Unable to read GITHUB_TOKEN. Check Colab Secrets "
        "and enable notebook access."
    ) from e

# ------------------------------------------------------------
# STEP 2: VERIFY GITHUB IDENTITY
# ------------------------------------------------------------

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

user_response = requests.get(
    "https://api.github.com/user",
    headers=headers,
    timeout=20
)

if user_response.status_code != 200:
    raise RuntimeError(
        f"GitHub authentication failed: HTTP "
        f"{user_response.status_code}"
    )

github_user = user_response.json()

github_login = github_user["login"]
github_id = github_user["id"]

print("GITHUB AUTHENTICATION SUCCESSFUL")
print("Authenticated account:", github_login)

# ------------------------------------------------------------
# STEP 3: VERIFY REPOSITORY ACCESS
# ------------------------------------------------------------

repo_response = requests.get(
    REPO_API,
    headers=headers,
    timeout=20
)

if repo_response.status_code != 200:
    raise RuntimeError(
        f"Repository access failed: HTTP "
        f"{repo_response.status_code}"
    )

repo_data = repo_response.json()
permissions = repo_data.get("permissions", {})

print("\nREPOSITORY:", repo_data["full_name"])
print("DEFAULT BRANCH:", repo_data["default_branch"])
print("PUSH PERMISSION:", permissions.get("push", "Not reported"))

if permissions.get("push") is False:
    raise PermissionError(
        "Your account does not have push permission."
    )

# ------------------------------------------------------------
# STEP 4: PREPARE TEMPORARY GIT AUTHENTICATION
# ------------------------------------------------------------

ASKPASS_PATH.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\n'
    'esac\n'
)

os.chmod(
    ASKPASS_PATH,
    stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR
)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(ASKPASS_PATH)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN

# ------------------------------------------------------------
# STEP 5: CLONE ONLY IF REPOSITORY DOES NOT EXIST
# ------------------------------------------------------------

if CLONE_PATH.exists():

    if not (CLONE_PATH / ".git").exists():
        raise RuntimeError(
            f"{CLONE_PATH} exists but is not a Git repository. "
            "No files were changed."
        )

    print("\nExisting repository found.")
    print("Reusing:", CLONE_PATH)

    existing_remote = subprocess.run(
        ["git", "remote", "get-url", "origin"],
        cwd=CLONE_PATH,
        capture_output=True,
        text=True
    )

    if existing_remote.returncode != 0:
        raise RuntimeError("Existing repository has no origin remote.")

    if existing_remote.stdout.strip() != REPO_URL:
        raise RuntimeError(
            "The existing folder points to a different repository. "
            "Stopping to prevent accidental changes."
        )

else:

    print("\nCloning repository...")

    clone_result = subprocess.run(
        ["git", "clone", REPO_URL, str(CLONE_PATH)],
        env=git_env,
        capture_output=True,
        text=True
    )

    if clone_result.returncode != 0:
        raise RuntimeError(
            "Clone failed:\n" + clone_result.stderr[-2000:]
        )

    print("Repository cloned successfully.")

# ------------------------------------------------------------
# STEP 6: CONFIGURE COMMIT IDENTITY
# ------------------------------------------------------------

subprocess.run(
    ["git", "config", "user.name", github_login],
    cwd=CLONE_PATH,
    check=True
)

noreply_email = (
    f"{github_id}+{github_login}@users.noreply.github.com"
)

subprocess.run(
    ["git", "config", "user.email", noreply_email],
    cwd=CLONE_PATH,
    check=True
)

# ------------------------------------------------------------
# STEP 7: FETCH REMOTE BRANCHES
# ------------------------------------------------------------

print("\nFetching remote branches...")

fetch_result = subprocess.run(
    ["git", "fetch", "origin"],
    cwd=CLONE_PATH,
    env=git_env,
    capture_output=True,
    text=True
)

if fetch_result.returncode != 0:
    raise RuntimeError(
        "Git fetch failed:\n" + fetch_result.stderr[-2000:]
    )

# ------------------------------------------------------------
# STEP 8: CHECK OUT OR CREATE YOUR BRANCH
# ------------------------------------------------------------

current_branch_result = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=CLONE_PATH,
    capture_output=True,
    text=True,
    check=True
)

current_branch = current_branch_result.stdout.strip()

remote_branch = subprocess.run(
    [
        "git", "show-ref", "--verify", "--quiet",
        f"refs/remotes/origin/{BRANCH_NAME}"
    ],
    cwd=CLONE_PATH
)

if current_branch == BRANCH_NAME:

    print(f"Already on branch: {BRANCH_NAME}")

elif remote_branch.returncode == 0:

    subprocess.run(
        ["git", "switch", "--track", f"origin/{BRANCH_NAME}"],
        cwd=CLONE_PATH,
        check=True
    )

    print(f"Switched to existing remote branch: {BRANCH_NAME}")

else:

    subprocess.run(
        ["git", "switch", "-c", BRANCH_NAME],
        cwd=CLONE_PATH,
        check=True
    )

    print(f"Created local branch: {BRANCH_NAME}")

# ------------------------------------------------------------
# STEP 9: REMOVE TEMPORARY AUTHENTICATION HELPER
# ------------------------------------------------------------

ASKPASS_PATH.unlink(missing_ok=True)

git_env.pop("GITHUB_TOKEN", None)
GITHUB_TOKEN = None
headers = None

# ------------------------------------------------------------
# STEP 10: VERIFY BRANCH AND STATUS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CURRENT BRANCH")
print("=" * 60)

print(run_git(["branch", "--show-current"]).stdout)

print("GIT STATUS")
print("=" * 60)

print(run_git(["status", "--short", "--branch"]).stdout)

print("REMOTE")
print("=" * 60)

print(run_git(["remote", "-v"]).stdout)

# ------------------------------------------------------------
# STEP 11: DISPLAY REPOSITORY STRUCTURE
# ------------------------------------------------------------

print("\nREPOSITORY FILE STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk(CLONE_PATH):

    dirs[:] = sorted(
        directory for directory in dirs
        if directory != ".git"
    )

    relative_root = Path(root).relative_to(CLONE_PATH)
    depth = len(relative_root.parts)

    if depth > 4:
        dirs[:] = []
        continue

    indent = "  " * depth

    for directory in dirs:
        print(f"{indent}[DIR]  {directory}/")

    for filename in sorted(files):
        print(f"{indent}[FILE] {filename}")

print("\n" + "=" * 60)
print("GITHUB SETUP VERIFIED")
print("Repository:", repo_data["full_name"])
print("Branch:", BRANCH_NAME)
print("No project files were copied or pushed.")
print("=" * 60)


In [ ]:

# ============================================================
# STEP 3: AUDIT PERSON 2 GOOGLE DRIVE FILES AND GITHUB TARGET
# Read-only: does not copy, edit, or delete anything.
# ============================================================

from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive if not already mounted
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

DRIVE_BASE = Path(
    "/content/drive/MyDrive/Sensory_Gift_Concierge/Person_2_AI"
)

GITHUB_BASE = Path(
    "/content/UnboxJoy"
)

print("=" * 65)
print("PERSON 2 GOOGLE DRIVE AUDIT")
print("=" * 65)

if not DRIVE_BASE.exists():
    print("ERROR: Person 2 Drive folder was not found.")
    print("Expected location:", DRIVE_BASE)
    print("\nAvailable folders under Sensory_Gift_Concierge:")

    parent = Path(
        "/content/drive/MyDrive/Sensory_Gift_Concierge"
    )

    if parent.exists():
        for item in sorted(parent.iterdir()):
            print(item.name)
    else:
        print("Sensory_Gift_Concierge folder does not exist.")

else:
    total_files = 0

    for root, dirs, files in os.walk(DRIVE_BASE):
        dirs[:] = sorted(
            d for d in dirs
            if d != ".ipynb_checkpoints"
        )

        relative = Path(root).relative_to(DRIVE_BASE)
        depth = len(relative.parts)
        indent = "  " * depth

        for directory in sorted(dirs):
            print(f"{indent}[DIR]  {directory}/")

        for filename in sorted(files):
            file_path = Path(root) / filename

            try:
                size_kb = file_path.stat().st_size / 1024
            except OSError:
                size_kb = 0

            print(
                f"{indent}[FILE] {filename} "
                f"({size_kb:.1f} KB)"
            )
            total_files += 1

    print("\nTotal files found:", total_files)

print("\n" + "=" * 65)
print("CURRENT GITHUB REPOSITORY")
print("=" * 65)

for item in sorted(GITHUB_BASE.iterdir()):
    if item.name == ".git":
        continue

    kind = "DIR " if item.is_dir() else "FILE"
    print(f"[{kind}] {item.name}")

print("\nAUDIT COMPLETE — NO FILES WERE MODIFIED.")


In [ ]:

# ============================================================
# STEP 4: LOCATE PERSON 2 COLAB NOTEBOOK
# Read-only Drive search
# ============================================================

from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

keywords = [
    "gift_concierge",
    "person2",
    "personalization",
    "sensory_gift",
    "unboxjoy"
]

print("=" * 65)
print("SEARCHING GOOGLE DRIVE FOR PERSON 2 NOTEBOOKS")
print("=" * 65)

matches = []

for notebook in MY_DRIVE.rglob("*.ipynb"):
    name = notebook.name.lower()

    if any(keyword in name for keyword in keywords):
        try:
            size_kb = notebook.stat().st_size / 1024
        except OSError:
            size_kb = 0

        matches.append((notebook, size_kb))

if matches:
    for index, (notebook, size_kb) in enumerate(matches, start=1):
        print(f"\n{index}. {notebook.name}")
        print("   Location:", notebook)
        print(f"   Size: {size_kb:.1f} KB")
else:
    print("No matching notebook found.")

print("\n" + "=" * 65)
print("PERSON 2 FOLDER — COMPLETE DIRECTORY DETAILS")
print("=" * 65)

PERSON2 = MY_DRIVE / "Sensory_Gift_Concierge" / "Person_2_AI"

if PERSON2.exists():
    for path in sorted(PERSON2.rglob("*")):
        if ".ipynb_checkpoints" in path.parts:
            continue

        relative = path.relative_to(PERSON2)

        if path.is_dir():
            print("[DIR] ", relative)
        elif path.is_file():
            print("[FILE]", relative)
else:
    print("Person 2 folder not found.")

print("\nSEARCH COMPLETED — NO FILES MODIFIED.")


In [ ]:

# ============================================================
# STEP 5: PERSON 2 NOTEBOOK AND FILE SECURITY AUDIT
# Read-only. Does not modify Drive or GitHub files.
# ============================================================

import json
import re
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/"
    "Gift_Concierge_Person2_AI_Personalization.ipynb"
)

PERSON2_DRIVE = Path(
    "/content/drive/MyDrive/Sensory_Gift_Concierge/Person_2_AI"
)

if not NOTEBOOK_PATH.exists():
    raise FileNotFoundError(f"Notebook not found: {NOTEBOOK_PATH}")

# ------------------------------------------------------------
# 1. LOAD NOTEBOOK
# ------------------------------------------------------------

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

print("=" * 65)
print("NOTEBOOK INFORMATION")
print("=" * 65)

print("Notebook:", NOTEBOOK_PATH.name)
print("Notebook format:", notebook.get("nbformat"))
print("Total cells:", len(cells))
print("Code cells:", sum(c.get("cell_type") == "code" for c in cells))
print("Markdown cells:", sum(c.get("cell_type") == "markdown" for c in cells))

# ------------------------------------------------------------
# 2. DISPLAY MARKDOWN HEADINGS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NOTEBOOK SECTION HEADINGS")
print("=" * 65)

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "markdown":
        continue

    source = cell.get("source", [])
    if isinstance(source, list):
        source = "".join(source)

    for line in source.splitlines():
        line = line.strip()
        if line.startswith("#"):
            print(f"Cell {index}: {line[:150]}")

# ------------------------------------------------------------
# 3. CHECK FOR POSSIBLE CREDENTIALS
# ------------------------------------------------------------

secret_patterns = {
    "Groq API key pattern": r"gsk_[A-Za-z0-9]{15,}",
    "Google API key pattern": r"AIza[0-9A-Za-z\-_]{20,}",
    "GitHub token pattern": r"gh[pousr]_[A-Za-z0-9]{20,}",
    "Generic API key assignment": (
        r"(?i)(api[_-]?key|access[_-]?token|secret)"
        r"\s*[:=]\s*['\"][^'\"]{12,}['\"]"
    )
}

findings = []

for cell_index, cell in enumerate(cells):
    source = cell.get("source", [])
    if isinstance(source, list):
        source = "".join(source)

    outputs_text = json.dumps(
        cell.get("outputs", []),
        ensure_ascii=False
    )

    combined = source + "\n" + outputs_text

    for label, pattern in secret_patterns.items():
        if re.search(pattern, combined):
            findings.append((cell_index, label))

print("\n" + "=" * 65)
print("CREDENTIAL SCAN")
print("=" * 65)

if findings:
    print("POSSIBLE CREDENTIALS DETECTED:")
    for cell_index, label in findings:
        print(f"Cell {cell_index}: {label}")
    print("\nDo not transfer the notebook until these are reviewed.")
else:
    print("No common hardcoded credential patterns detected.")
    print("Note: this is a pattern scan, not a guarantee of security.")

# ------------------------------------------------------------
# 4. CHECK NOTEBOOK OUTPUTS
# ------------------------------------------------------------

output_cells = []

for index, cell in enumerate(cells):
    outputs = cell.get("outputs", [])
    if outputs:
        output_cells.append((index, len(outputs)))

print("\n" + "=" * 65)
print("SAVED NOTEBOOK OUTPUTS")
print("=" * 65)

if output_cells:
    for index, count in output_cells:
        print(f"Cell {index}: {count} saved output(s)")
else:
    print("No saved outputs found.")

# ------------------------------------------------------------
# 5. LIST SUPPORTING FILES
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PERSON 2 SUPPORTING FILES")
print("=" * 65)

if PERSON2_DRIVE.exists():
    for path in sorted(PERSON2_DRIVE.rglob("*")):
        if not path.is_file():
            continue

        if ".ipynb_checkpoints" in path.parts:
            continue

        size_kb = path.stat().st_size / 1024
        relative = path.relative_to(PERSON2_DRIVE)

        print(f"{relative} ({size_kb:.1f} KB)")
else:
    print("Person 2 supporting folder not found.")

# ------------------------------------------------------------
# 6. CHECK NOTEBOOK FILE SIZE
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("AUDIT SUMMARY")
print("=" * 65)

print("Notebook size:", round(NOTEBOOK_PATH.stat().st_size / 1024, 2), "KB")
print("Credential-pattern findings:", len(findings))
print("Cells with saved outputs:", len(output_cells))
print("Audit completed. No files modified.")
